In [23]:
!git clone https://github.com/Sasha15151/MLF-Skin-Cancer-Classification.git

Cloning into 'MLF-Skin-Cancer-Classification'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 85 (delta 19), reused 42 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 22.44 KiB | 1.40 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [25]:
%cd MLF-Skin-Cancer-Classification
!ls

/content/drive/MyDrive/Skin Cancer Data/MLF-Skin-Cancer-Classification
notebooks  README.md  requirements.txt


In [26]:
!pip install -q -r requirements.txt

# Data Loading

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
%cd /content/drive/MyDrive/Skin Cancer Data

/content/drive/MyDrive/Skin Cancer Data


# Data Normalization

In [29]:
import pandas as pd

df = pd.read_csv("HAM10000_metadata.tab", sep="\t")

label_dict = {label: idx for idx, label in enumerate(df["dx"].unique())}
df["label"] = df["dx"].map(label_dict)

print(label_dict)

{'bkl': 0, 'nv': 1, 'df': 2, 'mel': 3, 'vasc': 4, 'bcc': 5, 'akiec': 6}


In [30]:
import os
from PIL import Image
from torch.utils.data import Dataset

class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"] + ".jpg"
        label = self.df.iloc[idx]["label"]

        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [31]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [32]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [33]:
from torch.utils.data import DataLoader

image_dir = "HAM10000_images"

train_dataset = SkinCancerDataset(train_df, image_dir, transform)
val_dataset   = SkinCancerDataset(val_df, image_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [34]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])
